# HW2 Part 1: Data Cleaning & Validation

Source table: `ohlc_hw` in `data/data.db` (copy the bCourses database into this folder's `data/` directory; it is gitignored).

In [1]:
import sqlite3
from contextlib import closing
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DB_PATH = Path("data/data.db")
assert DB_PATH.exists(), f"Database not found at {DB_PATH.resolve()} - download it from bCourses"

## 1.1 Load and Inspect the Data

In [2]:
# sqlite3's own `with` only commits/rolls back; closing() also closes the connection
with closing(sqlite3.connect(DB_PATH)) as conn:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

raw.info()
raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


### Column meanings

**ts** : the date of the observation (one trading day), written as YYYY-MM-DD. `.info()` shows it is stored as text (`object`), not as a date type

**open** : the raw (unadjusted) opening price of that trading day

**high** : the raw maximum price reached during that trading day

**low** : the raw lowest price reached during that trading day

**close** : the raw closing price of that trading day

**adj_close** : the closing price adjusted for stock splits and dividends, so returns stay comparable over time (on old dates it is lower than `close`)

**volume** : the number of shares traded that day

**ticker** : the representative symbol of the stock

In [3]:
# ts is ISO text (YYYY-MM-DD), so the text order is also the date order
date_range_text = f"The date range is from {raw['ts'].min()} to {raw['ts'].max()}"
print(date_range_text)

print(f"Distinct ticker labels: {raw['ticker'].nunique()}")
display(raw['ticker'].unique())

The date range is from 1962-01-02 to 2024-12-06
Distinct ticker labels: 150


array(['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ', 'DIS',
       '<a target="_blank"><span style="color: blue">CSCO</span></a>',
       'SHW', 'KO',
       '<a target="_blank"><span style="font-weight: bold; color: blue">HD</span></a>',
       '<a target="_blank"><span style="font-weight: bold; color: blue">VZ</span></a>',
       'IBM', 'PG', 'HON', 'AXP', 'MMM', 'GS', 'BA', 'CRM', 'UNH', 'NKE',
       'NVDA', 'AMGN', 'CSCO', 'MCD',
       '<a target="_blank"><span style="color: red">AMGN</span></a>',
       'VZ', 'V',
       '<a target="_blank"><span style="color: red">DIS</span></a>',
       '<a target="_blank"><span style="font-weight: bold; color: blue">NKE</span></a>',
       '<a target="_blank"><span style="color: blue">TRV</span></a>',
       'MRK', 'WMT',
       '<a target="_blank"><span style="font-weight: bold; color: red">SHW</span></a>',
       '<a target="_blank"><span style="font-weight: bold; color: red">JNJ</span></a>',
       '<a target="_blank"><span style="font-weight:

There are 150 distinct labels, but many are the same stock wrapped in HTML markup (e.g. `<span style="color: blue">KO</span>`); they are normalized to 30 tickers in 1.2.

## 1.2 Detect Data Quality Issues

### Approach

Every detector takes a DataFrame and returns rows in the **Data Quality Report** format
(`ticker | issue_type | issue_date | description | severity`), so they can simply be concatenated.
Detectors only *diagnose*: nothing here changes `raw`. Repairs happen in 1.3.

Order matters:
1. **Normalize tickers first**: the raw labels contain HTML markup, so grouping by the raw label would split one stock into several.
2. **Find exact duplicates** next. The remaining row-level checks run on a de-duplicated *view*, so a bad row that was copied 3 times is reported once, and repeated rows don't create fake 0% returns.
3. Run the missing-value, calendar, price, return, and volume checks.

**Severity policy** (how much damage the issue does if left alone, and how safely it can be fixed):

| Severity | Meaning | Issue types |
|---|---|---|
| high | Impossible or ambiguous value that would corrupt calculations; no safe automatic fix | missing raw price, invalid (nonpositive / non-finite) price, OHLC bound violation, conflicting `(ticker, ts)` key, invalid volume |
| medium | Probably wrong or missing, but a documented repair rule exists | ticker formatting, missing `adj_close`, `adj_close` spike, missing trading day, extreme return outlier (> 10 std) |
| low | Suspicious but harmless once handled, or just a review flag | exact duplicate, zero volume, return outlier (3–10 std) |

In [4]:
REPORT_COLUMNS = ["ticker", "issue_type", "issue_date", "description", "severity"]
PRICE_COLUMNS = ["open", "high", "low", "close"]               # raw OHLC prices
ALL_PRICE_COLUMNS = PRICE_COLUMNS + ["adj_close"]
VALUE_COLUMNS = ["ts"] + ALL_PRICE_COLUMNS + ["volume"]       # every column except the ticker label


def make_issues(rows: pd.DataFrame, issue_type: str, description, severity: str) -> pd.DataFrame:
    # Convert flagged observations (with `ticker` and `ts`) into Data Quality Report rows.
    # `description` is one string for every row, or a Series of row-specific text aligned with `rows`.
    if isinstance(description, pd.Series):
        description = description.to_numpy()
    return pd.DataFrame(
        {
            "ticker": rows["ticker"].to_numpy(),
            "issue_type": issue_type,
            "issue_date": rows["ts"].to_numpy(),
            "description": description,
            "severity": severity,
        },
        columns=REPORT_COLUMNS,
    )


def concat_issues(parts: list[pd.DataFrame]) -> pd.DataFrame:
    # Concatenate report pieces, skipping empty ones (a check can correctly find zero issues).
    parts = [p for p in parts if len(p)]
    if not parts:
        return pd.DataFrame(columns=REPORT_COLUMNS)
    return pd.concat(parts, ignore_index=True)

#### Ticker formatting

In [5]:
HTML_TAG = r"<[^>]+>"


def normalize_ticker(labels: pd.Series) -> pd.Series:
    """Remove HTML markup, surrounding quotes and whitespace, then upper-case.

    Example: '<a target="_blank"><span style="color: blue">KO</span></a>'  ->  KO
    """
    return (
        labels.str.replace(HTML_TAG, "", regex=True)
        .str.strip()
        .str.strip("'\"")
        .str.strip()
        .str.upper()
    )


def add_normalized_ticker(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with the original label kept in `raw_ticker` and a clean `ticker` column."""
    out = df.rename(columns={"ticker": "raw_ticker"})
    out["ticker"] = normalize_ticker(out["raw_ticker"])
    return out


def detect_ticker_formatting(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Audit raw labels and flag every label that differs from its normalized ticker.

    Returns (audit, issues). Formatting is a property of the label, not of one date,
    so there is one issue row per label: issue_date is left empty and the description
    gives the number of rows and their date span.
    """
    audit = (
        df.groupby(["ticker", "raw_ticker"])
        .agg(rows=("ts", "size"), first_ts=("ts", "min"), last_ts=("ts", "max"))
        .reset_index()
    )
    bad = audit[audit["raw_ticker"] != audit["ticker"]]
    description = (
        "label " + bad["raw_ticker"].map(repr) + " used on " + bad["rows"].astype(str)
        + " rows (" + bad["first_ts"] + " to " + bad["last_ts"] + ")"
    )
    issues = pd.DataFrame(
        {
            "ticker": bad["ticker"],
            "issue_type": "ticker_formatting",
            "issue_date": "",
            "description": description,
            "severity": "medium",
        },
        columns=REPORT_COLUMNS,
    )
    return audit, issues

#### Repeated records

In [6]:
def detect_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Flag exact repeated records and conflicting (ticker, ts) keys. Needs normalized tickers.

    - Exact duplicate: same ticker and identical values in every column. Each extra copy is
      flagged; the first copy is the one we keep.
    - Conflicting key: same (ticker, ts) but different values. Every version is flagged, because
      we cannot tell which one is right without investigating.
    """
    exact = df.duplicated(subset=["ticker"] + VALUE_COLUMNS, keep="first")
    exact_issues = make_issues(
        df[exact], "duplicate_record", "exact repeat of an earlier row (after ticker normalization)", "low"
    )

    unique = df[~exact]
    conflict = unique.duplicated(subset=["ticker", "ts"], keep=False)
    versions = unique.groupby(["ticker", "ts"])["ts"].transform("size")
    conflict_issues = make_issues(
        unique[conflict],
        "conflicting_key",
        versions[conflict].astype(str) + " different records share this (ticker, ts)",
        "high",
    )
    return concat_issues([exact_issues, conflict_issues])

#### Missing data

Missing **cells** (a row exists but a value is empty) and missing **trading days** (the whole row is absent) are different checks.

For the trading calendar we use the assignment's allowed reference: the dates in the clean `ohlc` table, **per ticker**, and only between that ticker's own first and last observation (so days before a stock listed don't count as "missing").
Business weekdays alone would wrongly count exchange holidays as gaps.

In [7]:
MISSING_SEVERITY = {
    "open": "high", "high": "high", "low": "high", "close": "high",  # raw prices: needed for everything
    "adj_close": "medium",  # needed for reports, but rebuildable from close and a trusted factor
    "volume": "low",
}


def summarize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Number and percentage of missing cells in each column."""
    return pd.DataFrame({"missing": df.isna().sum(), "pct": (df.isna().mean() * 100).round(3)})


def detect_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """One issue row per missing price or volume cell."""
    return concat_issues([
        make_issues(df[df[col].isna()], f"missing_{col}", f"{col} is missing", severity)
        for col, severity in MISSING_SEVERITY.items()
    ])


def load_reference_calendar(db_path: Path) -> pd.DataFrame:
    """Trading dates per ticker from the supplied `ohlc` table."""
    with closing(sqlite3.connect(db_path)) as conn:
        return pd.read_sql_query("SELECT DISTINCT ticker, ts FROM ohlc", conn)


def find_missing_trading_days(df: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    """Return (ticker, ts) pairs that are in the reference calendar but have no row in df.

    Only dates between each ticker's first and last observation in df are checked.
    """
    observed = df[["ticker", "ts"]].drop_duplicates()
    span = observed.groupby("ticker")["ts"].agg(first="min", last="max")
    expected = calendar.merge(span, left_on="ticker", right_index=True)
    expected = expected[(expected["ts"] >= expected["first"]) & (expected["ts"] <= expected["last"])]
    merged = expected[["ticker", "ts"]].merge(observed, on=["ticker", "ts"], how="left", indicator=True)
    missing = merged.loc[merged["_merge"] == "left_only", ["ticker", "ts"]]
    return missing.sort_values(["ticker", "ts"]).reset_index(drop=True)


def summarize_missing_trading_days(df: pd.DataFrame, missing: pd.DataFrame) -> list[str]:
    """One sentence per ticker: "Ticker X is missing Y trading days between [start] and [end]"."""
    span = df.groupby("ticker")["ts"].agg(["min", "max"])
    counts = missing.groupby("ticker").size().reindex(span.index, fill_value=0)
    return [
        f"Ticker {t} is missing {counts[t]} trading days between {span.loc[t, 'min']} and {span.loc[t, 'max']}"
        for t in span.index
    ]


def detect_missing_trading_days(missing: pd.DataFrame) -> pd.DataFrame:
    """One issue row per absent trading day (no prices are invented for it)."""
    return make_issues(missing, "missing_trading_day", "date is in the reference calendar but has no row", "medium")

#### Outliers and impossible prices

Returns are `(close - prev_close) / prev_close`, computed within each ticker after sorting by date.
The 3-standard-deviation screen uses each ticker's own mean and sample std (`ddof=1`) and flags both tails.
It is a **review list**: most big moves are real market events, so nothing is deleted because of it.

`adj_close` gets its own check. The adjustment factor `adj_close / close` only changes at dividends and splits, and those changes *persist*.
A factor that jumps away from the surrounding days and comes straight back is a one-day spike, i.e. a probable data-entry error.

In [8]:
def add_simple_returns(df: pd.DataFrame) -> pd.DataFrame:
    """Sort by ticker/date and add `ret` = (close - prev_close) / prev_close within each ticker.

    Needs one row per (ticker, ts): repeated rows would create fake 0% returns.
    """
    out = df.sort_values(["ticker", "ts"]).copy()
    out["ret"] = out.groupby("ticker")["close"].pct_change(fill_method=None)
    return out


EXTREME_STD = 10  # flags beyond this many std are rated medium instead of low


def detect_return_outliers(df: pd.DataFrame, n_std: float = 3.0) -> pd.DataFrame:
    """Flag abs(ret - mean) > n_std * std within each ticker (sample std, ddof=1), both tails.

    Undefined returns (a ticker's first day, NaN, or +/-inf) are excluded from the mean and std
    and never flagged. A ticker whose std is 0 or undefined (fewer than 2 returns) gets no flags.
    """
    ret = df["ret"].where(np.isfinite(df["ret"]))
    by_ticker = ret.groupby(df["ticker"])
    mean = by_ticker.transform("mean")
    std = by_ticker.transform("std", ddof=1)
    usable = ret.notna() & std.notna() & (std > 0)
    distance = (ret - mean).abs() / std
    flagged = usable & (distance > n_std)

    rows, dist = df[flagged], distance[flagged]
    description = (
        "return " + (rows["ret"] * 100).round(2).astype(str) + "% is "
        + dist.round(1).astype(str) + " std from the ticker mean"
    )
    severity = np.where(dist > EXTREME_STD, "medium", "low")
    issues = make_issues(rows, "return_outlier", description, "low")
    issues["severity"] = severity
    return issues


def detect_adj_close_spikes(df: pd.DataFrame, ratio: float = 2.0, window: int = 5) -> pd.DataFrame:
    """Flag one-day spikes in the adjustment factor adj_close / close.

    The factor is compared with the median factor of the `window` rows before AND the `window`
    rows after (within the ticker). It is flagged only when it is more than `ratio` times larger
    (or smaller) than BOTH sides. A real split or dividend changes the factor permanently, so one
    side still agrees and it is not flagged; using medians keeps two bad days in a row detectable.
    Rows with missing adj_close are skipped (they are reported as missing_adj_close).
    """
    df = df.sort_values(["ticker", "ts"])
    factor = df["adj_close"] / df["close"]
    factor = factor.where(np.isfinite(factor) & (factor > 0))

    def one_sided_median(s: pd.Series) -> pd.Series:
        return s.shift(1).rolling(window, min_periods=1).median()

    by_ticker = factor.groupby(df["ticker"])
    before = by_ticker.transform(one_sided_median)
    after = by_ticker.transform(lambda s: one_sided_median(s[::-1])[::-1])

    up = (factor > ratio * before) & (factor > ratio * after)
    down = (factor * ratio < before) & (factor * ratio < after)
    flagged = factor.notna() & (up | down)

    rows = df[flagged]
    typical = ((before + after) / 2)[flagged]
    description = (
        "adj_close/close = " + factor[flagged].round(4).astype(str)
        + " vs about " + typical.round(4).astype(str) + " on surrounding days"
    )
    return make_issues(rows, "adj_close_spike", description, "medium")


def detect_invalid_prices(df: pd.DataFrame) -> pd.DataFrame:
    """Flag prices that are <= 0 or infinite, in raw OHLC and adj_close (NaN counts as missing, not here)."""
    parts = []
    for col in ALL_PRICE_COLUMNS:
        rows = df[(df[col] <= 0) | np.isinf(df[col])]
        parts.append(make_issues(rows, "invalid_price", col + " = " + rows[col].astype(str) + " (must be positive and finite)", "high"))
    return concat_issues(parts)

#### Inconsistencies

Raw prices must satisfy `low <= open <= high` and `low <= close <= high`. Each broken rule is its own issue row, so one observation can have several.
`adj_close` is not checked against these bounds because it is on a different (adjusted) scale.

In [9]:
def detect_ohlc_violations(df: pd.DataFrame) -> pd.DataFrame:
    """Flag every broken OHLC relationship (one issue row per rule per observation)."""
    rules = {
        "high < close": df["high"] < df["close"],
        "low > close": df["low"] > df["close"],
        "high < open": df["high"] < df["open"],
        "low > open": df["low"] > df["open"],
        "low > high": df["low"] > df["high"],
    }
    parts = []
    for rule, broken in rules.items():
        rows = df[broken]
        values = (
            " (open=" + rows["open"].astype(str) + ", high=" + rows["high"].astype(str)
            + ", low=" + rows["low"].astype(str) + ", close=" + rows["close"].astype(str) + ")"
        )
        parts.append(make_issues(rows, "ohlc_violation", rule + values, "high"))
    return concat_issues(parts)


def detect_volume_issues(df: pd.DataFrame) -> pd.DataFrame:
    """Zero volume (possibly stale data) is low severity; negative or infinite volume is impossible."""
    zero = df[df["volume"] == 0]
    invalid = df[(df["volume"] < 0) | np.isinf(df["volume"])]
    return concat_issues([
        make_issues(zero, "zero_volume", "volume is 0 (possibly stale or untraded)", "low"),
        make_issues(invalid, "invalid_volume", "volume = " + invalid["volume"].astype(str), "high"),
    ])

#### Data Quality Report

In [10]:
SEVERITY_ORDER = pd.CategoricalDtype(["high", "medium", "low"], ordered=True)


def build_quality_report(raw_df: pd.DataFrame, calendar: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run every detector on a raw OHLC table.

    Returns (report, ticker_audit). raw_df itself is never modified.
    """
    labeled = add_normalized_ticker(raw_df)
    ticker_audit, ticker_issues = detect_ticker_formatting(labeled)
    duplicate_issues = detect_duplicates(labeled)

    # Row-level checks run on a de-duplicated view so each distinct observation is reported once.
    unique = labeled.drop_duplicates(subset=["ticker"] + VALUE_COLUMNS)
    # Returns need one row per (ticker, ts); conflicting keys are left out (they are reported above).
    one_per_key = unique[~unique.duplicated(subset=["ticker", "ts"], keep=False)]
    missing_days = find_missing_trading_days(unique, calendar)

    report = concat_issues([
        ticker_issues,
        duplicate_issues,
        detect_missing_values(unique),
        detect_missing_trading_days(missing_days),
        detect_invalid_prices(unique),
        detect_ohlc_violations(unique),
        detect_return_outliers(add_simple_returns(one_per_key)),
        detect_adj_close_spikes(one_per_key),
        detect_volume_issues(unique),
    ])
    report["severity"] = report["severity"].astype(SEVERITY_ORDER)
    report = report.sort_values(["ticker", "issue_date", "severity", "issue_type"]).reset_index(drop=True)
    return report, ticker_audit

In [11]:
calendar = load_reference_calendar(DB_PATH)
quality_report, ticker_audit = build_quality_report(raw, calendar)

print(f"{len(quality_report):,} issues found in {len(raw):,} raw rows\n")
quality_report.groupby(["issue_type", "severity"], observed=True).size().rename("count").to_frame()

63,500 issues found in 397,711 raw rows



count
issue_type        severity       
adj_close_spike   medium    10405
duplicate_record  low       29460
invalid_price     high          5
missing_adj_close medium    18374
ohlc_violation    high          5
return_outlier    medium       63
                  low        4993
ticker_formatting medium      120
zero_volume       low          75

In [12]:
# Ticker audit: 150 raw labels collapse to this many real tickers
labeled = add_normalized_ticker(raw)
print(f"{raw['ticker'].nunique()} raw labels -> {labeled['ticker'].nunique()} tickers")
ticker_audit.head(10)

150 raw labels -> 30 tickers


,ticker,raw_ticker,rows,first_ts,last_ts
0,AAPL,"<a target=""_blank""><span style=""color: blue"">A...",307,1981-10-19,2024-09-04
1,AAPL,"<a target=""_blank""><span style=""color: red"">AA...",318,1980-12-19,2024-09-06
2,AAPL,"<a target=""_blank""><span style=""font-weight: b...",306,1981-02-04,2024-09-26
3,AAPL,"<a target=""_blank""><span style=""font-weight: b...",264,1980-12-29,2024-07-19
4,AAPL,AAPL,10747,1980-12-12,2024-10-18
5,AMGN,"<a target=""_blank""><span style=""color: blue"">A...",294,1983-07-29,2024-08-14
6,AMGN,"<a target=""_blank""><span style=""color: red"">AM...",290,1983-07-15,2024-09-09
7,AMGN,"<a target=""_blank""><span style=""font-weight: b...",280,1983-08-30,2024-06-05
8,AMGN,"<a target=""_blank""><span style=""font-weight: b...",272,1983-07-07,2024-09-19
9,AMGN,AMGN,10154,1983-06-17,2024-10-18


In [13]:
# Missing cells per column (on the de-duplicated rows, so copies are not counted twice)
unique_view = labeled.drop_duplicates(subset=["ticker"] + VALUE_COLUMNS)
summarize_missing_values(unique_view)

,missing,pct
ts,0,0.00
open,0,0.00
high,0,0.00
low,0,0.00
close,0,0.00
adj_close,18374,4.99
volume,0,0.00
raw_ticker,0,0.00
ticker,0,0.00


In [14]:
missing_days = find_missing_trading_days(unique_view, calendar)
for sentence in summarize_missing_trading_days(unique_view, missing_days):
    print(sentence)

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

In [15]:
# A few examples of each issue type
quality_report.groupby("issue_type", observed=True).head(2).sort_values("issue_type")

,ticker,issue_type,issue_date,description,severity
9,AAPL,adj_close_spike,1981-02-03,adj_close/close = 10.2537 vs about 0.7709 on s...,medium
22,AAPL,adj_close_spike,1981-04-28,adj_close/close = 16.1485 vs about 0.7709 on s...,medium
5,AAPL,duplicate_record,1981-01-02,exact repeat of an earlier row (after ticker n...,low
6,AAPL,duplicate_record,1981-01-05,exact repeat of an earlier row (after ticker n...,low
15105,CVX,invalid_price,1962-01-02,open = 0.0 (must be positive and finite),high
28949,JNJ,invalid_price,1962-01-02,open = 0.0 (must be positive and finite),high
7,AAPL,missing_adj_close,1981-01-08,adj_close is missing,medium
8,AAPL,missing_adj_close,1981-01-13,adj_close is missing,medium
15106,CVX,ohlc_violation,1962-01-02,"low > open (open=0.0, high=3.3, low=3.24, clos...",high
28950,JNJ,ohlc_violation,1962-01-02,"low > open (open=0.0, high=0.22338, low=0.2222...",high


#### Cross-check: `adj_close` spikes vs the reference table

The spike detector only uses evidence inside `ohlc_hw`. As a sanity check, compare it with the supplied clean `ohlc` table:
an `adj_close` that disagrees with the reference is a known bad value.

In [16]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    reference = pd.read_sql_query("SELECT ticker, ts, adj_close AS adj_close_ref FROM ohlc", conn)

check = unique_view.merge(reference, on=["ticker", "ts"], how="left")
known_bad = check["adj_close"].notna() & ~np.isclose(check["adj_close"], check["adj_close_ref"], rtol=1e-6)
known_bad_keys = set(zip(check.loc[known_bad, "ticker"], check.loc[known_bad, "ts"]))

spikes = quality_report[quality_report["issue_type"] == "adj_close_spike"]
spike_keys = set(zip(spikes["ticker"], spikes["issue_date"]))

print(f"adj_close values that disagree with the reference: {len(known_bad_keys):,}")
print(f"flagged by the spike detector:                     {len(spike_keys):,}")
print(f"  of which really disagree (true positives):       {len(spike_keys & known_bad_keys):,}")
print(f"  disagreeing values the detector missed:          {len(known_bad_keys - spike_keys):,}")

adj_close values that disagree with the reference: 10,417
flagged by the spike detector:                     10,405
  of which really disagree (true positives):       10,404
  disagreeing values the detector missed:          13


#### Demonstration on a small injected example

Some required issues do not occur in this database (missing trading days, `high < close`, `low > close`,
nonpositive close or adjusted close, non-finite prices, conflicting keys). Here they are injected into a tiny
made-up table to show that each detector catches them.

In [17]:
demo_dates = ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08", "2024-01-09"]
demo_calendar = pd.DataFrame({"ticker": "DEMO", "ts": demo_dates})

demo = pd.DataFrame({
    "ts": demo_dates,
    "open": [10.0, 10.2, 10.1, 10.3, 10.4, 10.5],
    "high": [10.5, 10.6, 10.4, 10.6, 10.8, 10.9],
    "low": [9.8, 10.0, 9.9, 10.1, 10.2, 10.3],
    "close": [10.2, 10.1, 10.3, 10.4, 10.6, 10.7],
    "adj_close": [10.2, 10.1, 10.3, 10.4, 10.6, 10.7],
    "volume": [1000, 1100, 900, 1200, 0, 1000],
    "ticker": [" demo ", "DEMO", "DEMO", "<b>DEMO</b>", "DEMO", "DEMO"],
})
demo = demo.drop(index=2)                         # missing trading day: 2024-01-04 removed
demo.loc[1, "high"] = 9.0                         # high < close (and high < open, low > high)
demo.loc[3, "close"] = -1.0                       # negative close (also low > close)
demo.loc[4, "adj_close"] = np.inf                 # non-finite adjusted close
demo = pd.concat([
    demo,
    demo.iloc[[0]],                                # exact duplicate of the first row
    demo.iloc[[4]].assign(close=10.65, adj_close=10.65),  # conflicting record for 2024-01-09
], ignore_index=True)

demo_report, _ = build_quality_report(demo, demo_calendar)
demo_report

,ticker,issue_type,issue_date,description,severity
0,DEMO,ticker_formatting,,label ' demo ' used on 2 rows (2024-01-02 to 2...,medium
1,DEMO,ticker_formatting,,label '<b>DEMO</b>' used on 1 rows (2024-01-05...,medium
2,DEMO,duplicate_record,2024-01-02,exact repeat of an earlier row (after ticker n...,low
3,DEMO,ohlc_violation,2024-01-03,"high < close (open=10.2, high=9.0, low=10.0, c...",high
4,DEMO,ohlc_violation,2024-01-03,"high < open (open=10.2, high=9.0, low=10.0, cl...",high
5,DEMO,ohlc_violation,2024-01-03,"low > high (open=10.2, high=9.0, low=10.0, clo...",high
6,DEMO,missing_trading_day,2024-01-04,date is in the reference calendar but has no row,medium
7,DEMO,invalid_price,2024-01-05,close = -1.0 (must be positive and finite),high
8,DEMO,ohlc_violation,2024-01-05,"low > close (open=10.3, high=10.6, low=10.1, c...",high
9,DEMO,invalid_price,2024-01-08,adj_close = inf (must be positive and finite),high


## 1.3 Clean the Data

### Cleaning decisions

| Issue (from the 1.2 report) | Decision | Why |
|---|---|---|
| HTML-marked ticker labels | Normalize; keep the original label in `raw_ticker` | Required before grouping/deduplicating |
| Exact duplicate records | Remove (keep the first copy) | Identical values, so nothing is lost |
| Conflicting `(ticker, ts)` records | Quarantine every version | We cannot tell which version is right |
| All four raw prices missing | Drop the row, **before** filling | Nothing to repair from |
| Missing raw price | Forward-fill within the ticker and flag it | Assignment rule ("carry last known price") |
| Missing price with no earlier day (leading gap) | Quarantine | Filling would mean extrapolating |
| Nonpositive / non-finite raw price, or broken OHLC bounds (e.g. the 5 `open = 0` rows) | **Quarantine** | All 5 are a ticker's first row: there is no previous day to fill from, and inventing an open from that day's other prices would be a guess |
| Forward-filled prices that break today's range | Quarantine | A filled close can fall outside today's high/low |
| Missing volume | Fill with 0 and flag it | Assignment rule; the flag records that volume was *unknown*, not zero |
| One-day round trips (price jumps > 30% and is undone the next day: TRV 1976-10-28, 1978-12-20, 1981-08-10) | **Interpolate** open/high/low/close **and adj_close** as the midpoint of the day before and after; flag it | Confirmed data error: the whole row doubles and returns to the old level the next day. The `ohlc` reference has the same error on these dates, so its adjustment factor is not used there |
| Other return outliers (> 3 std) | **Keep and flag** | Every one matches the trusted `ohlc` prices exactly and coincides with real market events; news was not verified row by row |
| Missing or spiked `adj_close` | Rebuild as `close × (ohlc.adj_close / ohlc.close)` on the same `(ticker, ts)`; flag it | Trusted, date-specific adjustment factor (source: supplied `ohlc` table). A row with no reference match would be quarantined |

Repair flags are saved with the cleaned table; quarantined rows are saved with their reason.

In [18]:
def bounds_violation(df: pd.DataFrame) -> pd.Series:
    """True where present raw prices break low <= open <= high or low <= close <= high.

    Missing values never count as a violation here (they are handled by filling).
    """
    return (
        (df["low"] > df["open"]) | (df["open"] > df["high"])
        | (df["low"] > df["close"]) | (df["close"] > df["high"])
        | (df["low"] > df["high"])
    )


def quarantine(df: pd.DataFrame, mask: pd.Series, reason: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split df into (kept rows, quarantined rows); quarantined rows get a `quarantine_reason`."""
    return df[~mask], df[mask].assign(quarantine_reason=reason)


def remove_exact_duplicates(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Drop exact repeats of an earlier row (needs normalized tickers). Returns (df, rows removed)."""
    repeated = df.duplicated(subset=["ticker"] + VALUE_COLUMNS, keep="first")
    return df[~repeated], int(repeated.sum())


def drop_all_prices_missing(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Drop rows where open, high, low and close are ALL missing. Returns (df, rows removed)."""
    empty = df[PRICE_COLUMNS].isna().all(axis=1)
    return df[~empty], int(empty.sum())

In [19]:
def forward_fill_prices(df: pd.DataFrame) -> pd.DataFrame:
    """Forward-fill missing raw prices within each ticker; df must be sorted by ticker and ts.

    Adds `flag_price_ffill`. A leading gap (no earlier price) stays missing and is
    quarantined by the caller.
    """
    out = df.copy()
    was_missing = out[PRICE_COLUMNS].isna().any(axis=1)
    out[PRICE_COLUMNS] = out.groupby("ticker")[PRICE_COLUMNS].ffill()
    out["flag_price_ffill"] = was_missing & out[PRICE_COLUMNS].notna().all(axis=1)
    return out


def fill_missing_volume(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing volume with 0 and add `flag_volume_filled` (unknown volume is not proof of no trading)."""
    out = df.copy()
    out["flag_volume_filled"] = out["volume"].isna()
    out["volume"] = out["volume"].fillna(0)
    return out

In [20]:
def find_round_trips(df: pd.DataFrame, min_move: float = 0.30, tolerance: float = 0.03) -> pd.Series:
    """True for rows whose close jumps by more than `min_move` and is undone the next day.

    "Undone" means the next close is within `tolerance` of the previous close. A row needs a
    valid day before AND after to qualify, so leading/trailing rows are never interpolated.
    df must be sorted by ticker and ts.
    """
    close = df.groupby("ticker")["close"]
    prev_close, next_close = close.shift(1), close.shift(-1)
    jump = (df["close"] / prev_close - 1).abs() > min_move
    undone = (next_close / prev_close - 1).abs() < tolerance
    return jump & undone


def interpolate_round_trips(df: pd.DataFrame, errors: pd.Series) -> pd.DataFrame:
    """Replace OHLC and adj_close on confirmed error rows with the midpoint of the neighbouring days.

    For a single bad day, the midpoint is linear interpolation. Because both neighbours satisfy
    the OHLC bounds, their midpoint does too. Run this after rebuild_adj_close, so the neighbours'
    adj_close is already repaired. Adds `flag_round_trip_interpolated`.
    """
    out = df.copy()
    for col in ALL_PRICE_COLUMNS:
        by_ticker = out.groupby("ticker")[col]
        midpoint = (by_ticker.shift(1) + by_ticker.shift(-1)) / 2
        out.loc[errors, col] = midpoint[errors]
    out["flag_round_trip_interpolated"] = errors
    return out

In [21]:
def rebuild_adj_close(df: pd.DataFrame, reference: pd.DataFrame, rtol: float = 1e-6) -> tuple[pd.DataFrame, pd.Series]:
    """Rebuild adj_close = close x trusted factor, where factor = ohlc.adj_close / ohlc.close.

    The factor is matched on the exact (ticker, ts), never carried across dates. A row is rebuilt when
    its adj_close is missing (`flag_adj_filled`) or disagrees with close x factor, i.e. the spikes
    (`flag_adj_replaced`). Returns (df, unresolved): unresolved rows
    needed a rebuild but have no reference match.
    """
    factors = reference.assign(factor=reference["adj_close"] / reference["close"])[["ticker", "ts", "factor"]]
    out = df.merge(factors, on=["ticker", "ts"], how="left", validate="one_to_one")

    expected = out["close"] * out["factor"]
    has_factor = out["factor"].notna()
    missing = out["adj_close"].isna()
    disagrees = ~missing & has_factor & ~np.isclose(out["adj_close"], expected, rtol=rtol)

    out.loc[(missing | disagrees) & has_factor, "adj_close"] = expected
    out["flag_adj_filled"] = missing & has_factor
    out["flag_adj_replaced"] = disagrees
    unresolved = missing & ~has_factor
    return out.drop(columns="factor"), unresolved


def flag_return_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """Add `flag_return_outlier` using the 1.2 detector (3 std, per ticker). The rows are kept."""
    issues = detect_return_outliers(add_simple_returns(df))
    flagged = pd.MultiIndex.from_frame(issues[["ticker", "issue_date"]])
    out = df.copy()
    out["flag_return_outlier"] = pd.MultiIndex.from_frame(out[["ticker", "ts"]]).isin(flagged)
    return out

In [22]:
FLAG_COLUMNS = [
    "flag_price_ffill", "flag_volume_filled", "flag_round_trip_interpolated",
    "flag_adj_filled", "flag_adj_replaced", "flag_return_outlier",
]
CLEAN_COLUMNS = ["ticker", "ts"] + ALL_PRICE_COLUMNS + ["volume", "raw_ticker"] + FLAG_COLUMNS


def clean_ohlc(raw_df: pd.DataFrame, reference: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """Apply every cleaning step in order. raw_df is never modified.

    Returns (cleaned, quarantined, accounting) where accounting counts what happened to every raw row.
    """
    counts = {"raw rows": len(raw_df)}
    held = []  # quarantined pieces

    df = add_normalized_ticker(raw_df)
    df["ts"] = pd.to_datetime(df["ts"]).dt.strftime("%Y-%m-%d")    # consistent ISO dates

    df, counts["removed: exact duplicates"] = remove_exact_duplicates(df)
    df, q = quarantine(df, df.duplicated(["ticker", "ts"], keep=False), "conflicting (ticker, ts) records"); held.append(q)
    df = df.sort_values(["ticker", "ts"]).reset_index(drop=True)
    df, counts["removed: all raw prices missing"] = drop_all_prices_missing(df)

    # Bad values are quarantined BEFORE filling, so they are never carried forward.
    bad_value = ((df[PRICE_COLUMNS] <= 0) | np.isinf(df[PRICE_COLUMNS])).any(axis=1)
    df, q = quarantine(df, bad_value, "nonpositive or non-finite raw price"); held.append(q)
    df, q = quarantine(df, bounds_violation(df), "raw OHLC bounds violated"); held.append(q)

    df = forward_fill_prices(df)
    df, q = quarantine(df, df[PRICE_COLUMNS].isna().any(axis=1), "leading gap: no earlier price to carry forward"); held.append(q)
    df, q = quarantine(df, bounds_violation(df), "forward-filled prices violate today's OHLC bounds"); held.append(q)
    df = fill_missing_volume(df)

    df, unresolved = rebuild_adj_close(df, reference)
    df, q = quarantine(df, unresolved, "adj_close missing and no trusted reference factor"); held.append(q)
    df = interpolate_round_trips(df, find_round_trips(df))
    df = flag_return_outliers(df)

    cleaned = df[CLEAN_COLUMNS].reset_index(drop=True)
    quarantined = pd.concat([q for q in held if len(q)] or [pd.DataFrame(columns=["quarantine_reason"])], ignore_index=True)
    counts["quarantined"] = len(quarantined)
    counts["retained (ohlc_hw_cleaned)"] = len(cleaned)
    accounting = pd.Series(counts, name="rows")
    assert accounting["raw rows"] == accounting.drop("raw rows").sum(), "every raw row must be accounted for"
    return cleaned, quarantined, accounting

#### Run the cleaning

In [23]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    reference = pd.read_sql_query("SELECT ticker, ts, close, adj_close FROM ohlc", conn)

raw_before = raw.copy()
cleaned, quarantined, accounting = clean_ohlc(raw, reference)
assert raw.equals(raw_before), "the raw table must stay unchanged"

accounting.to_frame()

,rows
raw rows,397711
removed: exact duplicates,29460
removed: all raw prices missing,0
quarantined,5
retained (ohlc_hw_cleaned),368246


In [24]:
# Why rows were quarantined, and how many rows each repair touched
display(quarantined.groupby("quarantine_reason").size().rename("rows").to_frame())
display(quarantined[["ticker", "ts", "open", "high", "low", "close", "quarantine_reason"]])
cleaned[FLAG_COLUMNS].sum().rename("rows flagged").to_frame()

,rows
quarantine_reason,
nonpositive or non-finite raw price,5


,ticker,ts,open,high,low,close,quarantine_reason
0,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,nonpositive or non-finite raw price
1,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,nonpositive or non-finite raw price
2,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,nonpositive or non-finite raw price
3,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,nonpositive or non-finite raw price
4,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,nonpositive or non-finite raw price


,rows flagged
flag_price_ffill,0
flag_volume_filled,0
flag_round_trip_interpolated,3
flag_adj_filled,18374
flag_adj_replaced,10417
flag_return_outlier,5176


In [25]:
# The three interpolated round trips, with the days around them
fixed = cleaned.index[cleaned["flag_round_trip_interpolated"]]
around = sorted({i + k for i in fixed for k in (-1, 0, 1)})
cleaned.loc[around, ["ticker", "ts", "open", "high", "low", "close", "adj_close", "flag_round_trip_interpolated"]]

,ticker,ts,open,high,low,close,adj_close,flag_round_trip_interpolated
318326,TRV,1976-10-27,4.610,4.610,4.610,4.610,0.980460,False
318327,TRV,1976-10-28,4.650,4.650,4.650,4.650,0.988765,True
318328,TRV,1976-10-29,4.690,4.690,4.690,4.690,0.997070,False
318867,TRV,1978-12-19,4.020,4.020,4.020,4.020,0.906450,False
318868,TRV,1978-12-20,4.055,4.055,4.055,4.055,0.915270,True
318869,TRV,1978-12-21,4.090,4.090,4.090,4.090,0.924090,False
319532,TRV,1981-08-07,6.030,6.030,6.030,6.030,1.570000,False
319533,TRV,1981-08-10,6.030,6.030,6.030,6.030,1.570000,True
319534,TRV,1981-08-11,6.030,6.030,6.030,6.030,1.570000,False


#### Validation

In [26]:
def validate_cleaned(df: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    """Run the assignment's checks on a cleaned table; every row of the result should be True.

    Interpolated round-trip rows are left out of the adj_close/reference comparison on purpose:
    the reference shares their error, so their adj_close comes from the neighbouring days instead.
    """
    prices = df[ALL_PRICE_COLUMNS]
    adj_check = df[~df["flag_round_trip_interpolated"]].merge(reference, on=["ticker", "ts"], how="inner", suffixes=("", "_ref"))
    trusted_adj = adj_check["close"] * adj_check["adj_close_ref"] / adj_check["close_ref"]
    checks = {
        "no missing raw or adjusted prices": prices.notna().all().all(),
        "all prices finite and positive": (np.isfinite(prices) & (prices > 0)).all().all(),
        "unique (ticker, ts) keys": not df.duplicated(["ticker", "ts"]).any(),
        "volume finite and nonnegative": bool((np.isfinite(df["volume"]) & (df["volume"] >= 0)).all()),
        "low <= open/close <= high": not bounds_violation(df).any(),
        "adj_close consistent with trusted factor": bool(np.isclose(adj_check["adj_close"], trusted_adj, rtol=1e-6).all()),
        "ts stored as YYYY-MM-DD": bool(df["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()),
    }
    return pd.Series(checks, name="passed").to_frame()


validation = validate_cleaned(cleaned, reference)
assert validation["passed"].all(), validation
validation

,passed
no missing raw or adjusted prices,True
all prices finite and positive,True
"unique (ticker, ts) keys",True
volume finite and nonnegative,True
low <= open/close <= high,True
adj_close consistent with trusted factor,True
ts stored as YYYY-MM-DD,True


In [27]:
# Before/after statistics
stats = ["count", "mean", "min", "max"]
before = raw[ALL_PRICE_COLUMNS + ["volume"]].describe().T[stats]
after = cleaned[ALL_PRICE_COLUMNS + ["volume"]].describe().T[stats]
display(pd.concat({"before (raw)": before, "after (cleaned)": after}, axis=1))

# Rerun the 1.2 Data Quality Report on the cleaned data
report_after, _ = build_quality_report(cleaned[raw.columns], calendar)
pd.concat(
    {
        "before": quality_report["issue_type"].value_counts(),
        "after": report_after["issue_type"].value_counts(),
    },
    axis=1,
).fillna(0).astype(int)

before (raw)                                       after (cleaned)  \
                 count          mean       min           max           count   
open          397711.0  4.628548e+01  0.000000  6.042600e+02        368246.0   
high          397711.0  4.674766e+01  0.005208  6.086300e+02        368246.0   
low           397711.0  4.581628e+01  0.004801  5.978800e+02        368246.0   
close         397711.0  4.629290e+01  0.005208  6.054000e+02        368246.0   
adj_close     377856.0  5.165225e+01  0.002945  1.044251e+04        368246.0   
volume        397711.0  3.097773e+07  0.000000  9.276606e+09        368246.0   

                                                 
                   mean       min           max  
open       4.630935e+01  0.005208  6.042600e+02  
high       4.677178e+01  0.005208  6.086300e+02  
low        4.584034e+01  0.004801  5.978800e+02  
close      4.631729e+01  0.005208  6.054000e+02  
adj_close  3.727465e+01  0.002945  6.054000e+02  
volume     3.095184e+07  0.000000  9.276606e+09

,before,after
issue_type,,
duplicate_record,29460,0
missing_adj_close,18374,0
adj_close_spike,10405,0
return_outlier,5056,5176
ticker_formatting,120,0
zero_volume,75,74
invalid_price,5,0
ohlc_violation,5,0


After cleaning, the only issues left are the ones we deliberately keep: return outliers (kept and flagged) and zero-volume days (reported, not changed).
There are slightly more return outliers than before: removing TRV's three ±100% round trips lowers TRV's standard deviation, so more of its ordinary large days now exceed 3 std.

#### Save `ohlc_hw_cleaned`

In [28]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    with conn:  # commits both tables together
        cleaned.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
        quarantined.to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False)
    saved_rows = pd.read_sql_query("SELECT COUNT(*) AS n FROM ohlc_hw_cleaned", conn)["n"].iloc[0]

assert saved_rows == len(cleaned)
print(f"Saved {saved_rows:,} rows to ohlc_hw_cleaned and {len(quarantined)} rows to ohlc_hw_quarantine in {DB_PATH}")

Saved 368,246 rows to ohlc_hw_cleaned and 5 rows to ohlc_hw_quarantine in data\data.db


#### Demonstration on a small injected example

Several repair cases do not occur in this database: missing prices, missing volume, a row with all prices missing,
a leading gap, broken OHLC bounds, and a forward-fill that breaks today's range. This tiny made-up table has
one of each (plus a round trip, a missing `adj_close` and a spiked `adj_close`), and goes through the same `clean_ohlc`.

In [29]:
dates = pd.bdate_range("2024-01-01", periods=10).strftime("%Y-%m-%d")
base = np.array([10.0, 10.1, 10.2, 10.25, 10.3, 20.6, 10.35, 11.0, 10.5, 10.6])  # day 5: the whole row doubles
demo_raw = pd.DataFrame({
    "ts": dates, "open": base - 0.05, "high": base + 0.2, "low": base - 0.2, "close": base,
    "adj_close": base * 0.5, "volume": 1000.0, "ticker": "DEMO",
})
demo_reference = pd.DataFrame({"ticker": "DEMO", "ts": dates, "close": base, "adj_close": base * 0.5})
demo_reference.loc[5, ["close", "adj_close"]] = [20.6, 10.3]       # the reference has the same round trip

demo_raw.loc[0, "open"] = np.nan                                    # leading gap -> quarantine
demo_raw.loc[2, "high"] = 10.1                                      # high < close -> quarantine
demo_raw.loc[3, "close"] = np.nan                                   # missing close -> forward-fill
demo_raw.loc[4, "adj_close"] = np.nan                               # missing adj_close -> rebuild
demo_raw.loc[6, "adj_close"] = 51.75                                # spiked adj_close -> replace
demo_raw.loc[7, "close"] = np.nan                                   # filled close (10.35) < low (10.8) -> quarantine
demo_raw.loc[8, PRICE_COLUMNS] = np.nan                             # all prices missing -> drop
demo_raw.loc[9, "volume"] = np.nan                                  # missing volume -> 0 + flag

demo_clean, demo_quarantine, demo_accounting = clean_ohlc(demo_raw, demo_reference)
display(demo_accounting.to_frame())
display(demo_quarantine[["ts", "open", "high", "low", "close", "quarantine_reason"]])
display(demo_clean.drop(columns=["raw_ticker", "flag_return_outlier"]))
validate_cleaned(demo_clean, demo_reference)

,rows
raw rows,10
removed: exact duplicates,0
removed: all raw prices missing,1
quarantined,3
retained (ohlc_hw_cleaned),6


,ts,open,high,low,close,quarantine_reason
0,2024-01-03,10.15,10.1,10.0,10.20,raw OHLC bounds violated
1,2024-01-01,NaN,10.2,9.8,10.00,leading gap: no earlier price to carry forward
2,2024-01-10,10.95,11.2,10.8,10.35,forward-filled prices violate today's OHLC bounds


,ticker,ts,open,high,low,close,adj_close,volume,flag_price_ffill,flag_volume_filled,flag_round_trip_interpolated,flag_adj_filled,flag_adj_replaced
0,DEMO,2024-01-02,10.050,10.300,9.900,10.100,5.0500,1000.0,False,False,False,False,False
1,DEMO,2024-01-04,10.200,10.450,10.050,10.100,5.0500,1000.0,True,False,False,False,True
2,DEMO,2024-01-05,10.250,10.500,10.100,10.300,5.1500,1000.0,False,False,False,True,False
3,DEMO,2024-01-08,10.275,10.525,10.125,10.325,5.1625,1000.0,False,False,True,False,False
4,DEMO,2024-01-09,10.300,10.550,10.150,10.350,5.1750,1000.0,False,False,False,False,True
5,DEMO,2024-01-12,10.550,10.800,10.400,10.600,5.3000,0.0,False,True,False,False,False


,passed
no missing raw or adjusted prices,True
all prices finite and positive,True
"unique (ticker, ts) keys",True
volume finite and nonnegative,True
low <= open/close <= high,True
adj_close consistent with trusted factor,True
ts stored as YYYY-MM-DD,True
